# 栄養と1食テーブル

担当：高居
入力: `data/prices.csv`、`data/recipes.csv`、`data/raw/mext_food_composition_raw.csv`（成分表から読み取った値）
入力（もう1つ）: `data/raw/dri.csv`（食事摂取基準2025年版・18〜29歳の値）
出力: `data/nutrition.csv`（食材ごとの栄養。team_rules の `nutrients.csv` に当たる）、`data/meals.csv`（1食テーブル。最適化の唯一の入口。team_rules 第2節の列すべて）、`data/nutrition_target.csv`（参考：食事摂取基準から作った1日の目標値）

※ `src/` から実行する（`optimize.ipynb` と同じ。データは `../data/`）。
※ `meals.csv` の外食の行は王さんの担当。このノートブックは自炊の行だけを作り直し、すでにある外食の行（今は岩間さんのダミー m01〜m05）はそのまま残す。

## 第4回でやったこと（口頭説明用）

**目的**　外食と自炊を比べて、1日3食の組み合わせを最適化するには、自炊の1食ごとの「コスト・時間・栄養」が要る。このノートブックは、それを `meals.csv`（team_rules 第2節）の形で作る。

**データ元**　文部科学省「日本食品標準成分表（八訂）増補2023年」の値（100gあたり、可食部）。文科省の「食品成分データベース」の画面から、食材ごとに読み取った。

**手順**
1. 価格の22品目それぞれに、成分表のどの食品が当たるかを決める（例：豆腐 → 木綿豆腐）。
2. 9項目（エネルギー・たんぱく質・脂質・炭水化物・食物繊維・食塩相当量・カルシウム・鉄・ビタミンC）を読み取り、`data/raw/` に保存する。
3. 値の書き方を整える。`(0)` のようにカッコ付きは推定値、`Tr` は微量、`-` は未測定。数値にして、カッコ付き・Tr があったかを記録する。→ `nutrition.csv`
4. レシピ（`recipes.csv`）の使用量から、1食の栄養を足し合わせる。
5. 同じレシピの材料費（最新月の価格）と、調理＋片付け時間を付ける。材料費は `base_cost_yen`（ロス前）、`cost_yen = base_cost_yen ÷ (1 − ロス率)`（ロス率は `config.LOSS_RATE`。未決の間は 0.1 で仮置き。`sensitivity.ipynb` の基準と同じ）。→ `meals.csv`
6. 食事摂取基準（厚生労働省）から、1日の栄養の目標値を作る。→ `nutrition_target.csv`（性別・身体活動レベルごと。最適化の栄養条件に使う）

**考え方（仮定）**
- レシピの `amount_g` は「買う量（廃棄する部分を含む）」とみなす。成分表の値は可食部100gあたりなので、`amount_g × (1 − 廃棄率) × 値 ÷ 100` で栄養を出す。材料費は買う量で計算するので、廃棄ぶんは材料費に入っている。
- レシピは分析用の仮想レシピ。調理時間・片付け時間は仮の値なので、`is_assumed = true` にしてある。

**課題**　調理パンは成分表で該当食品が見つからず未登録（どのレシピにも使っていない）。目標値は男女どちらを使うか（または両方選べるようにするか）をチームで決める。

**push 前に「Clear All Outputs」を実行する**

> **このセル**：入力・出力ファイルの場所、gへの換算値、扱う栄養9項目を決めている。
>
> **聞かれたら**：9項目のうち、エネルギー・たんぱく質・脂質・炭水化物は最適化の制約に使う。食物繊維・食塩・Ca・鉄・VitCは表示だけ。

In [ ]:
# ===== 設定セル：ファイルの場所と、g への換算に使う仮の値 =====
from pathlib import Path        # ファイルの場所（パス）を扱う

import pandas as pd             # 表（DataFrame）を扱う

import config                   # 共通の仮定値（src/config.py）。ロス率 LOSS_RATE を使う

# --- 入力ファイル ---
PRICE_PATH = Path("../data/prices.csv")        # 価格表（prices.ipynb が作る）
RECIPE_PATH = Path("../data/recipes.csv")      # レシピ表（recipes.ipynb が作る）
# （ノートブックは src/ から実行する。データは ../data/）
# 成分表から読み取った生の値。カッコ付きの値や Tr など、文字のまま保存してある
RAW_NUT_PATH = Path("../data/raw/mext_food_composition_raw.csv")

# 食事摂取基準（2025年版・18〜29歳）。公式資料の値を手で写したもの。列: item, sex, kind, activity_level, value, unit, note
DRI_PATH = Path("../data/raw/dri.csv")

# --- 出力ファイル ---
OUT_NUT_PATH = Path("../data/nutrition.csv")                # 食材ごとの栄養（100gあたり、数値に整えたもの）
OUT_MEALS_PATH = Path("../data/meals.csv")                  # 1食テーブル（team_rules 第2節の列すべて。補助5項目も含む）
OUT_TARGET_PATH = Path("../data/nutrition_target.csv")      # 1日の栄養の目標値（参考。最適化の制約は config.py の KCAL_TABLE・ENERGY_RATIO から作る）

# g への換算（仮定値）。recipes.ipynb と同じ値にそろえる。あとで src/config.py に移す
EGG_G_PER_PIECE = 60   # 鶏卵 1個 = 60g（殻つきの目安。廃棄率14%で可食部は約52g）
MILK_G_PER_ML = 1.0    # 1mL = 1g として扱う（牛乳）

# 栄養9項目：{nutrition.csv の列名: 日本語名と単位}。この順番で出力する
NUTRIENTS = {
    "energy_kcal": "エネルギー(kcal)", "protein_g": "たんぱく質(g)", "fat_g": "脂質(g)",
    "carb_g": "炭水化物(g)", "fiber_g": "食物繊維総量(g)", "salt_g": "食塩相当量(g)",
    "calcium_mg": "カルシウム(mg)", "iron_mg": "鉄(mg)", "vitamin_c_mg": "ビタミンC(mg)",
}

> **このセル**：成分表の値を数字に直す関数（ここは定義だけ）。
>
> **聞かれたら**：成分表では `(0)` が推定値、`Tr` が微量、`-` が未測定。推定値と微量は数字にして印を付け、未測定は空欄のままにしている。

In [ ]:
# ===== 関数セル1：成分表の値を数値に整える（定義だけ） =====

def parse_nutrient(value):
    """成分表の値（文字）を数値に直す。

    入力: value は "77.6" / "(0)" / "Tr" / "-" / 空欄 など。
        - "(0)" のようなカッコ付き … 推定値や類推値。カッコを外して数値にする（推定フラグを立てる）
        - "Tr" "(Tr)" … 微量（trace）。0 として扱う（推定フラグを立てる）
        - "-" や空欄 … 未測定。数値にできないので NaN（空欄）にする
    戻り値: (数値 or NaN, 推定値だったか True/False)
    """
    s = "" if pd.isna(value) else str(value).strip()
    if s in ("", "-"):
        return float("nan"), False             # 未測定は補完しない（空欄のまま）
    estimated = s.startswith("(") or "Tr" in s
    s = s.strip("()")
    if s == "Tr":
        return 0.0, estimated                  # 微量は 0 として扱う
    return float(s), estimated


def load_nutrition(raw_path):
    """成分表から読み取った CSV を読み、数値に整えた表（nutrition.csv の形）にする。

    入力: raw_path は mext_food_composition_raw.csv。item_id は prices.csv と同じ品目番号。
    戻り値: DataFrame（item_id, food_no, food_name, waste_pct, 栄養9列, has_estimated）
        has_estimated … 9項目のどれかが推定値（カッコ付き・Tr）なら True
    """
    # item_id と食品番号は「先頭の0」が消えないよう、文字として読む
    raw = pd.read_csv(raw_path, dtype=str, encoding="utf-8")
    out = raw[["item_id", "food_no", "food_name"]].copy()
    out["waste_pct"] = pd.to_numeric(raw["waste_pct"], errors="coerce")    # 廃棄率（%）。皮・殻・根などを除く割合
    flags = pd.Series(False, index=raw.index)
    for col in NUTRIENTS:
        parsed = raw[col].map(parse_nutrient)                              # (数値, 推定か) の組
        out[col] = parsed.map(lambda p: p[0])
        flags = flags | parsed.map(lambda p: p[1])
    out["has_estimated"] = flags
    return out.reset_index(drop=True)

> **このセル**：レシピ×価格×栄養を結んで、レシピごとに材料費・時間・栄養を合計する関数（ここは定義だけ）。
>
> **聞かれたら**：栄養＝買う量×(1−廃棄率)×100gあたりの値÷100。皮や根など食べない部分を引いている。準備時間＝調理＋片付け。base_cost_yen は材料費そのまま、cost_yen はロス率を入れた値（base÷(1−ロス率)）。

In [ ]:
# ===== 関数セル2：1食の栄養・材料費・時間を計算する（定義だけ） =====
# （grams_in_unit と price_per_gram は recipes.ipynb と同じ。ノートブック間で変数を共有しないため、ここにも置く）

def grams_in_unit(unit, egg_g_per_piece, ml_g_per_ml):
    """prices.csv の unit（価格がカバーする量）を g に直す。例: "5kg"->5000, "100g"->100, "10個"->10個ぶん, "450mL"->mL×換算。

    入力: unit は "5kg" "900g" "10個" "1000mL" のような文字列。空欄・未知の形式は NaN を返す。
    """
    import re
    m = re.fullmatch(r"(\d+(?:\.\d+)?)(kg|g|個|mL)", str(unit))
    if not m:
        return float("nan")
    n, u = float(m.group(1)), m.group(2)
    return {"kg": n * 1000, "g": n, "個": n * egg_g_per_piece, "mL": n * ml_g_per_ml}[u]


def price_per_gram(prices_df, egg_g_per_piece, ml_g_per_ml):
    """prices.csv の価格を 1gあたり（円/g）に換算する。価格 ÷ （unit を g にした量）。"""
    out = prices_df.copy()
    out["grams"] = out["unit"].map(lambda u: grams_in_unit(u, egg_g_per_piece, ml_g_per_ml))
    out["yen_per_g"] = out["price_yen"] / out["grams"]
    return out[["item_id", "name", "month", "yen_per_g"]]


def build_meals(recipes_df, prices_df, nutrition_df, month, egg_g_per_piece, ml_g_per_ml, loss_rate):
    """自炊レシピから、1食テーブル（meals.csv の自炊の行）を作る。

    入力: recipes_df は recipes.csv（recipe_id, name, ingredient, amount_g, cook_min, cleanup_min）。
          prices_df は prices.csv。nutrition_df は load_nutrition の戻り値。month は材料費に使う月（例 "2026-08"）。
          loss_rate は自炊のロス率（0.1 = 10%）。cost_yen = base_cost_yen ÷ (1 − loss_rate)。
    戻り値: dict
        {"meals": DataFrame（team_rules 第2節 meals.csv の列：meal_id, name, type, base_cost_yen, cost_yen, prep_min,
                   kcal, protein_g, fat_g, carb_g, fiber_g, salt_g, calcium_mg, iron_mg, vitamin_c_mg, exclude_slots, is_assumed）,
         "missing": 栄養または価格が付かなかった食材のリスト}
    """
    # 食材名（prices.csv の name）から item_id を引けるようにして、レシピの各行に item_id を付ける
    name_to_id = prices_df.drop_duplicates("item_id").set_index("name")["item_id"]
    df = recipes_df.copy()
    df["item_id"] = df["ingredient"].map(name_to_id)

    # 指定した月の 1gあたり価格を付ける → 食材ごとの材料費（円）= 使用量(g) × 円/g
    ppg = price_per_gram(prices_df, egg_g_per_piece, ml_g_per_ml)
    ppg = ppg[ppg["month"] == month][["item_id", "yen_per_g"]]
    df = df.merge(ppg, on="item_id", how="left")
    df["yen"] = df["amount_g"] * df["yen_per_g"]

    # 栄養を付ける。可食部の量 = 買う量 × (1 − 廃棄率)。成分値は 100g あたりなので ÷100
    df = df.merge(nutrition_df.drop(columns=["food_no", "food_name", "has_estimated"]), on="item_id", how="left")
    edible_g = df["amount_g"] * (1 - df["waste_pct"] / 100)
    for col in NUTRIENTS:
        df[col] = edible_g * df[col] / 100

    # 栄養も価格も付かなかった食材（計算から漏れているので、結果を見るときに注意する）
    missing = sorted(df[df["yen"].isna() | df["energy_kcal"].isna()]["ingredient"].unique())

    # レシピ（1食）ごとに合計する。料理名・時間は各レシピで同じなので先頭の値を使う
    g = df.groupby("recipe_id")
    total = g[["yen"] + list(NUTRIENTS)].sum(min_count=1)
    info = g[["name", "cook_min", "cleanup_min"]].first()
    t = info.join(total).reset_index()

    meals = pd.DataFrame({
        "meal_id": t["recipe_id"],
        "name": t["name"],
        "type": "自炊",
        "base_cost_yen": t["yen"].round(0),                  # ロス前の材料費（買う量ベース。廃棄ぶん込み）
        "cost_yen": t["yen"].round(0) / (1 - loss_rate),     # ロス率を入れた1食のコスト。sensitivity の apply_loss と同じ式（丸めない）
        "prep_min": t["cook_min"] + t["cleanup_min"],        # 準備時間 = 調理 + 片付け（team_rules 第2節）
        "kcal": t["energy_kcal"].round(0),
        "protein_g": t["protein_g"].round(1),
        "fat_g": t["fat_g"].round(1),
        "carb_g": t["carb_g"].round(1),
        "fiber_g": t["fiber_g"].round(1),
        "salt_g": t["salt_g"].round(1),
        "calcium_mg": t["calcium_mg"].round(0),
        "iron_mg": t["iron_mg"].round(1),
        "vitamin_c_mg": t["vitamin_c_mg"].round(0),
        "exclude_slots": "",                                  # 除外する枠なし（例外の食事だけ手で入れる）
        "is_assumed": True,                                   # 調理時間が仮の値・レシピが仮想なので true
    })
    return {"meals": meals, "missing": missing}


def check_meals(meals_df):
    """meals.csv の検算。問題点の文のリストを返す（空なら問題なし）。"""
    problems = []
    cols = ["cost_yen", "prep_min", "kcal", "protein_g", "fat_g", "carb_g"]
    if meals_df[cols].isna().any().any():
        problems.append("値が空欄の列がある（栄養や価格が付かなかった食材がある）")
    if (meals_df["kcal"] < 100).any() or (meals_df["kcal"] > 1500).any():
        problems.append("エネルギーが 100〜1500kcal の範囲外の1食がある（量や換算を見直す）")
    if meals_df["meal_id"].duplicated().any():
        problems.append("meal_id が重複している")
    return problems

> **このセル**：成分表の値を数字に整えて、未登録の品目と推定値を含む品目を確認する。
>
> **聞かれたら**：調理パンは成分表に当てはまる食品がなく未登録。どのレシピにも使っていない。

In [ ]:
# ===== 実行セル1：成分表の値を数値に整えて、中身を確認する =====
nutrition = load_nutrition(RAW_NUT_PATH)
print("品目数:", len(nutrition), "/ 栄養が入っている品目:", nutrition["energy_kcal"].notna().sum())
# 栄養が空欄の品目（成分表で見つからなかったもの）
print("栄養が未登録の品目:", nutrition[nutrition["energy_kcal"].isna()]["item_id"].tolist())
# 推定値（カッコ付き・Tr）を含む品目。使うときは「推定を含む」と分かるようにしておく
print("推定値を含む品目:", nutrition[nutrition["has_estimated"]]["item_id"].tolist())
nutrition

> **このセル**：食材ごとの栄養表を `data/nutrition.csv` に保存する。
>
> **聞かれたら**：どの食品を当てたかは `food_no`・`food_name` 列で分かる（例：豆腐→木綿豆腐）。

In [ ]:
# ===== 実行セル2：栄養表を data/nutrition.csv に保存する =====
# 品目名（prices.csv の name）を付けて、人が見て分かるようにしてから保存する
prices_for_name = pd.read_csv(PRICE_PATH, dtype={"item_id": str}).drop_duplicates("item_id")[["item_id", "name"]]
nutrition_out = prices_for_name.merge(nutrition, on="item_id", how="left")
nutrition_out.to_csv(OUT_NUT_PATH, index=False, encoding="utf-8")
print("保存しました:", OUT_NUT_PATH)

> **このセル**：1食テーブルを作る。自炊の行だけ作り直して、外食の行はそのまま残す。
>
> **聞かれたら**：ロス率はまだ未決なので、感度分析の基準に合わせて10%で仮置きしている。外食の行は王さんの担当で、今は岩間さんのダミー5件。

In [ ]:
# ===== 実行セル3：1食テーブル（meals.csv）を作って検算する =====
prices = pd.read_csv(PRICE_PATH, dtype={"item_id": str})
recipes = pd.read_csv(RECIPE_PATH)
latest = sorted(prices["month"].unique())[-1]                # 材料費は最新月の価格で計算する
# ロス率は config.LOSS_RATE。未決（None）の間は 0.1（10%）で仮置きする。
#   sensitivity.ipynb の基準（OAT の base["loss_rate"]=0.1）と検算「apply_loss(meals, 0.1) が cost_yen と一致」に合わせた値
loss_rate = config.LOSS_RATE if config.LOSS_RATE is not None else 0.1
result = build_meals(recipes, prices, nutrition, latest, EGG_G_PER_PIECE, MILK_G_PER_ML, loss_rate)
cooked = result["meals"]
print("材料費に使った月:", latest, "/ ロス率:", loss_rate)
print("栄養・価格が付かなかった食材:", result["missing"] or "なし")

# 外食の行（王さん担当。今は岩間さんのダミー）は、いまの meals.csv からそのまま残す
current = pd.read_csv(OUT_MEALS_PATH, dtype={"exclude_slots": str}, keep_default_na=False, na_values=[""])
eatout = current[current["type"] == "外食"]
meals = pd.concat([eatout, cooked], ignore_index=True)[list(cooked.columns)]
print("外食の行:", len(eatout), "/ 自炊の行:", len(cooked))
print("検算:", check_meals(meals) or "問題なし")
meals

> **このセル**：「豆腐とねぎのみそ汁ごはん」のkcalを、手計算と照合する。
>
> **聞かれたら**：手計算：米513＋豆腐73＋ねぎ6.3＋みそ27.3＝約620kcal。プログラムの結果と一致した。

In [ ]:
# ===== 実行セル4：手計算との照合（計算が合っているかの自己チェック） =====
# R006「豆腐とねぎのみそ汁ごはん」のエネルギーを、成分表の値から手で計算して、meals と比べる。
#   米 150g × 342kcal/100g                         = 513.0
#   豆腐 100g × 73kcal/100g                        =  73.0
#   ねぎ 30g（廃棄率40%→可食部18g）× 35kcal/100g    =   6.3
#   みそ 15g × 182kcal/100g                        =  27.3
hand = 150 * 3.42 + 100 * 0.73 + 30 * 0.6 * 0.35 + 15 * 1.82
got = float(meals.loc[meals["meal_id"] == "R006", "kcal"].iloc[0])
print(f"手計算 {hand:.1f} kcal / meals {got:.0f} kcal")
assert abs(hand - got) < 1, "手計算と合わない：計算を見直す"
print("一致しました")

> **このセル**：`meals.csv` を保存する。
>
> **聞かれたら**：これが岩間さんの最適化が読む唯一のファイル。列は team_rules 第2節の17列すべて。

In [ ]:
# ===== 保存セル：meals.csv を保存する =====
meals.to_csv(OUT_MEALS_PATH, index=False, encoding="utf-8")   # 最適化の唯一の入口（team_rules 第2節の列すべて）
print("保存しました:", OUT_MEALS_PATH)

> **このセル**：食事摂取基準から、1日の栄養の目標値を作る関数（ここは定義だけ）。
>
> **聞かれたら**：脂質と炭水化物は「エネルギーの何%」で決まっているので、gに直している（脂質1g＝9kcal、炭水化物1g＝4kcal）。

In [ ]:
# ===== 関数セル3：食事摂取基準から「1日の栄養の目標値」を作る（定義だけ） =====

def make_nutrition_target(dri_df, sex, activity_level):
    """食事摂取基準（18〜29歳）から、1日の栄養の目標値を dict にする。最適化（optimize）の nutrition_target に渡す形。

    入力: dri_df は dri.csv の DataFrame（item, sex, kind, activity_level, value）。
          sex は "male" / "female"。activity_level は "低い" / "ふつう" / "高い"（身体活動レベル I / II / III）。
    戻り値: dict（1日あたり。単位は列名のとおり）
        kcal            … 推定エネルギー必要量
        protein_g_min   … たんぱく質の推奨量
        fat_g_min/max   … 脂質の目標量（エネルギーの20〜30%を g に直す。脂質は 1g = 9kcal）
        carb_g_min/max  … 炭水化物の目標量（エネルギーの50〜65%を g に直す。炭水化物は 1g = 4kcal）
        fiber_g_min     … 食物繊維の目標量（以上）
        salt_g_max      … 食塩相当量の目標量（未満）
        calcium_mg_min / iron_mg_min / vitamin_c_mg_min … 推奨量
    """
    d = dri_df[dri_df["sex"] == sex]

    def pick(item, kind, level=None):
        """条件に合う行の値を1つ取り出す（無ければエラーにして、黙って間違えない）。"""
        rows = d[(d["item"] == item) & (d["kind"] == kind)]
        if level is not None:
            rows = rows[rows["activity_level"] == level]
        return float(rows["value"].iloc[0])

    kcal = pick("energy_kcal", "estimated_need", activity_level)
    return {
        "kcal": kcal,
        "protein_g_min": pick("protein_g", "rda"),
        "fat_g_min": kcal * pick("fat_pct", "dg_min") / 100 / 9,
        "fat_g_max": kcal * pick("fat_pct", "dg_max") / 100 / 9,
        "carb_g_min": kcal * pick("carb_pct", "dg_min") / 100 / 4,
        "carb_g_max": kcal * pick("carb_pct", "dg_max") / 100 / 4,
        "fiber_g_min": pick("fiber_g", "dg_min"),
        "salt_g_max": pick("salt_g", "dg_max"),
        "calcium_mg_min": pick("calcium_mg", "rda"),
        "iron_mg_min": pick("iron_mg", "rda"),
        "vitamin_c_mg_min": pick("vitamin_c_mg", "rda"),
    }

> **このセル**：男女×活動レベル3段階の目標値を `nutrition_target.csv` に保存する。
>
> **聞かれたら**：例：男性・ふつう＝2,600kcal。食事摂取基準2025年版の18〜29歳の値。

In [ ]:
# ===== 実行セル5：食事摂取基準から、男女の1日の目標値を作って保存する =====
dri = pd.read_csv(DRI_PATH, encoding="utf-8")
rows = []
for sex in ["male", "female"]:
    for level in ["低い", "ふつう", "高い"]:
        target = make_nutrition_target(dri, sex, level)
        rows += [(sex, level, key, round(val, 1)) for key, val in target.items()]
nutrition_target = pd.DataFrame(rows, columns=["sex", "activity_level", "key", "value"])
nutrition_target.to_csv(OUT_TARGET_PATH, index=False, encoding="utf-8")
print("保存しました:", OUT_TARGET_PATH)
# 身体活動レベルが「ふつう」の目標値を、男女並べて表示する（性別・活動量は最適化で選べるようにする想定）
nutrition_target[nutrition_target["activity_level"] == "ふつう"].pivot(index="key", columns="sex", values="value")

## 出典・確認メモ

- 出典：文部科学省「日本食品標準成分表（八訂）増補2023年」（文科省「食品成分データベース」で2026年10月に閲覧。データベースは2026年3月27日公表の正誤表に対応済みと書かれている）。画面には「試験的に公開している」とある。
- 画面と資料には「日本食品標準成分表（八訂）増補2023年から引用（又は出典）」「…を加工して作成」と書く。
- 利用条件：文科省の「日本食品標準成分表」のページ（2026年10月確認）に、食品成分データは自由に利用できる、二次利用では「日本食品標準成分表（八訂）増補2023年から引用（又は出典）」と書く、とある。
- 食事摂取基準：厚生労働省「日本人の食事摂取基準（2025年版）」の18〜29歳の値。出版社が載せている要約PDF2点で確認した（厚労省の原本は未確認）。鉄の推奨量（男7.0・女10.0mg。女性は月経ありの値）は1点でしか確認できていない。食物繊維は男20g以上・女18g以上、食塩相当量は男7.5g未満・女6.5g未満。
- 食品の選び方（どの食品を当てたか）は `nutrition.csv` の `food_no` / `food_name` で分かる。当て方は推定を含む（例：さけ＝刺身用サーモン → たいせいようさけ・養殖・皮つき・生、食用油＝キャノーラ油 → なたね油、鶏肉＝ブロイラーもも肉 → 若どりもも・皮つき・生）。
- 調理パン（サンドイッチ）は成分表で該当食品が見つからず未登録。
- カッコ付きの値（推定値）と Tr（微量→0）は `has_estimated = true` で分かる。
- 炭水化物は成分表の「炭水化物」（八訂のエネルギー計算に使う値）。
- 補助5項目の基準値（`nutrition_target.csv` の fiber_g_min など）は、`config.AUX_REFERENCE`（いまは未設定）に入れる候補。共有ファイルなので、入れるときは先にチャットで伝える。
- 量・時間・レシピは仮のもの。実測や根拠に差し替えるまで `is_assumed = true`。